# 粒子轨迹提取工作流
本笔记演示如何从 ToT 与 ToA 探测器文本数据中提取粒子轨迹并保存为独立的 100×100 矩阵。

## 1. 数据准备与路径配置
设置输入目录与 ToT、ToA 轨迹输出目录，并创建批量处理所需的文件结构。

In [49]:
from pathlib import Path

# TODO: update these paths以匹配实际的输入与输出位置
INPUT_DIR = Path(r"d:\Project\Timepix\OriginalData\Alpha\60")
OUTPUT_TOT_DIR = Path(r"d:\Project\Timepix\ProcessData\Alpha\60\ToT")
OUTPUT_TOA_DIR = Path(r"d:\Project\Timepix\ProcessData\Alpha\60\ToA")

for path in (OUTPUT_TOT_DIR, OUTPUT_TOA_DIR):
    path.mkdir(parents=True, exist_ok=True)

TARGET_MATRIX_SIZE = 100  # 输出矩阵尺寸
TOA_SCALE = 10000  # ToA 数值缩放因子

## 2. 批量读取 ToT 与 ToA 原始矩阵
遍历输入目录，按照文件名规则配对 ToT/ToA 文本并加载为 256×256 数值矩阵。

In [50]:
import numpy as np
from collections import defaultdict
from typing import Dict, List, Tuple


def strip_suffix(value: str, suffix: str) -> str:
    return value[: -len(suffix)] if value.endswith(suffix) else value


def normalize_stem(stem: str) -> str:
    base = stem.replace("_ToT_", "_").replace("_ToA_", "_")
    base = strip_suffix(base, "_ToT")
    base = strip_suffix(base, "_ToA")
    return base


def list_detector_pairs(input_dir: Path) -> Dict[str, Dict[str, Path]]:
    pairs: Dict[str, Dict[str, Path]] = defaultdict(dict)
    for txt_path in sorted(input_dir.rglob("*.txt")):
        stem = txt_path.stem
        if "_ToT" in stem:
            pairs[normalize_stem(stem)]["tot"] = txt_path
        elif "_ToA" in stem:
            pairs[normalize_stem(stem)]["toa"] = txt_path
    return pairs


def load_detector_matrices(tot_path: Path, toa_path: Path) -> Tuple[np.ndarray, np.ndarray]:
    tot = np.loadtxt(tot_path, dtype=np.float32)
    toa = np.loadtxt(toa_path, dtype=np.float32)
    if tot.shape != (256, 256) or toa.shape != (256, 256):
        raise ValueError(f"Unexpected matrix shape: {tot.shape} / {toa.shape}")
    tot = np.rint(tot).astype(np.int32)
    return tot, toa

## 3. 轨迹连通区域检测
基于 8 邻域对激活像素进行连通区域标记，记录每个粒子轨迹的像素坐标与边界信息。

In [51]:
from collections import deque

NEIGHBOR_OFFSETS = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1), (0, 1),
    (1, -1), (1, 0), (1, 1),
]


def extract_components(tot_grid: np.ndarray, toa_grid: np.ndarray) -> List[Dict[str, object]]:
    if tot_grid.shape != toa_grid.shape:
        raise ValueError("ToT and ToA grids must share the same shape")

    mask = (tot_grid > 0) | (toa_grid > 0)
    rows, cols = mask.shape
    visited = np.zeros_like(mask, dtype=bool)
    components: List[Dict[str, object]] = []

    for r in range(rows):
        for c in range(cols):
            if not mask[r, c] or visited[r, c]:
                continue

            queue: deque[Tuple[int, int]] = deque([(r, c)])
            coords: List[Tuple[int, int]] = []
            touches_edge = False
            row_min, row_max = rows, -1
            col_min, col_max = cols, -1

            while queue:
                rr, cc = queue.popleft()
                if visited[rr, cc]:
                    continue
                visited[rr, cc] = True
                if not mask[rr, cc]:
                    continue

                coords.append((rr, cc))
                if rr == 0 or rr == rows - 1 or cc == 0 or cc == cols - 1:
                    touches_edge = True

                row_min = min(row_min, rr)
                row_max = max(row_max, rr)
                col_min = min(col_min, cc)
                col_max = max(col_max, cc)

                for dr, dc in NEIGHBOR_OFFSETS:
                    nr, nc = rr + dr, cc + dc
                    if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc] and mask[nr, nc]:
                        queue.append((nr, nc))

            components.append(
                {
                    "coords": coords,
                    "touches_edge": touches_edge,
                    "row_min": row_min,
                    "row_max": row_max,
                    "col_min": col_min,
                    "col_max": col_max,
                }
            )

    return components

## 4. 轨迹几何特征筛选
依据像素数量、长短轴比与边缘触碰情况筛选出形态完整的粒子轨迹。
1. MIN_PIXEL_COUNT = 15:
- 表示轨迹中激活像素的最小数量。
- 如果轨迹的激活像素数少于该值，则认为轨迹过小，可能是噪声或无效数据，会被剔除。
2. MAX_AXIS_RATIO = 0.8:
- 表示轨迹的长短轴比的最大值。
- 轨迹的长轴和短轴分别是其外接矩形的高度和宽度，长短轴比定义为:
$$r = \frac{\min(\text{height}, \text{width})}{\max(\text{height}, \text{width})}$$
- 如果长短轴比大于该值，说明轨迹接近圆形或椭圆形，可能是无效数据，会被剔除。

3. MIN_LONGEST_SPAN = 6:
- 表示轨迹的最长边（长轴）的最小值。
- 如果轨迹的长轴小于该值，说明轨迹过短，可能是噪声或无效数据，会被剔除。

In [52]:
MIN_PIXEL_COUNT = 15 # 最小的激活像素数
MAX_AXIS_RATIO = 0.3 # 最大的长宽轴比
MIN_LONGEST_SPAN = 5 # 最小的轨迹长跨度


def component_metrics(component: Dict[str, object]) -> Dict[str, float]:
    coords: List[Tuple[int, int]] = component["coords"]  # type: ignore[index]
    pixel_count = len(coords)
    height = component["row_max"] - component["row_min"] + 1  # type: ignore[index]
    width = component["col_max"] - component["col_min"] + 1  # type: ignore[index]
    longest = max(height, width)
    shortest = min(height, width)
    axis_ratio = shortest / longest if longest else 1.0
    return {
        "pixel_count": pixel_count,
        "height": height,
        "width": width,
        "axis_ratio": axis_ratio,
        "longest_span": longest,
    }


def is_component_valid(
    component: Dict[str, object],
    min_pixels: int = MIN_PIXEL_COUNT,
    max_axis_ratio: float = MAX_AXIS_RATIO,
    min_longest_span: int = MIN_LONGEST_SPAN,
) -> bool:
    metrics = component_metrics(component)
    if component["touches_edge"]:  # type: ignore[index]
        return False
    if metrics["pixel_count"] < min_pixels:
        return False
    if metrics["longest_span"] < min_longest_span:
        return False
    if metrics["axis_ratio"] < max_axis_ratio:
        return False
    return True

## 5. 轨迹居中与 100×100 填充
计算轨迹质心，将连通区域平移至 100×100 窗口中心，并对 ToA 数值缩放成整数。

In [53]:
def shift_coords_to_center(coords: List[Tuple[int, int]], target_size: int = TARGET_MATRIX_SIZE) -> List[Tuple[int, int]]:
    """Shift coordinate list so that its centroid is at the center of a target_size x target_size canvas.

    - Uses integer shift based on rounded centroid to ensure integer indices
    - Raises ValueError if any shifted coordinate would go out of bounds
    """
    if not coords:
        raise ValueError("Empty coordinates for component")

    rows = np.array([r for r, _ in coords], dtype=np.float64)
    cols = np.array([c for _, c in coords], dtype=np.float64)

    # centroid (row, col)
    cy = float(rows.mean())
    cx = float(cols.mean())

    center = (target_size - 1) / 2.0
    dy = int(round(center - cy))
    dx = int(round(center - cx))

    shifted: List[Tuple[int, int]] = []
    for r, c in coords:
        rr = r + dy
        cc = c + dx
        if rr < 0 or rr >= target_size or cc < 0 or cc >= target_size:
            raise ValueError("Shifted coordinate out of target bounds")
        shifted.append((rr, cc))
    return shifted


def scale_toa_values(values: List[float], scale: int = TOA_SCALE) -> np.ndarray:
    """Scale ToA float values to integers using the given scale.

    - NaN/Inf will be converted to 0
    - Values are rounded to nearest integer and returned as int64
    """
    arr = np.asarray(values, dtype=np.float64)
    if arr.size == 0:
        return np.zeros((0,), dtype=np.int64)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    scaled = np.rint(arr * float(scale)).astype(np.int64)
    return scaled


def build_output_name(base_stem: str, index: int, channel: str) -> str:
    """Build output filename for a track.

    Example: base_stem='1_r0000', channel='ToA', index=1 -> '1_r0000_ToA_0001.txt'
    """
    return f"{base_stem}_{channel}_{index:04d}.txt"


def save_matrix(matrix: np.ndarray, path: Path) -> None:
    """Save an integer matrix to text file, ensuring parent directory exists."""
    path.parent.mkdir(parents=True, exist_ok=True)
    # Use integer formatting for int dtypes; fallback to generic if needed
    if np.issubdtype(matrix.dtype, np.integer):
        np.savetxt(path, matrix, fmt="%d")
    else:
        np.savetxt(path, matrix)


In [54]:
def render_component_matrix(
    coords: List[Tuple[int, int]],
    values: np.ndarray,
    target_size: int = TARGET_MATRIX_SIZE,
    dtype: np.dtype = np.int64,  # Updated to np.int64 to handle larger integers
) -> np.ndarray:
    shifted = shift_coords_to_center(coords, target_size)
    canvas = np.zeros((target_size, target_size), dtype=dtype)
    for (row, col), value in zip(shifted, values):
        canvas[row, col] = value
    return canvas

## 6. ToT/ToA 轨迹结果命名与保存
为每个有效轨迹生成统一的 ToT/ToA 文件名，将 100×100 轨迹矩阵写入目标目录。

In [55]:
def process_detector_directory(
    input_dir: Path,
    output_tot_dir: Path,
    output_toa_dir: Path,
    *,
    target_size: int = TARGET_MATRIX_SIZE,
    toa_scale: int = TOA_SCALE,
) -> Dict[str, int]:
    stats = {
        "paired_files": 0,
        "saved_tracks": 0,
        "skipped_tracks": 0,
        "missing_channels": 0,
    }

    file_pairs = list_detector_pairs(input_dir)
    for base_key, channels in file_pairs.items():
        if "tot" not in channels or "toa" not in channels:
            stats["missing_channels"] += 1
            continue

        tot_path = channels["tot"]
        toa_path = channels["toa"]
        stats["paired_files"] += 1

        tot_grid, toa_grid = load_detector_matrices(tot_path, toa_path)
        components = extract_components(tot_grid, toa_grid)

        rel_dir = tot_path.parent.relative_to(input_dir)
        base_stem = normalize_stem(tot_path.stem)

        for index, component in enumerate(components, start=1):
            if not is_component_valid(component):
                stats["skipped_tracks"] += 1
                continue

            coords: List[Tuple[int, int]] = component["coords"]  # type: ignore[index]
            tot_values = np.asarray([int(tot_grid[row, col]) for row, col in coords], dtype=np.int32)
            toa_values_raw = [float(toa_grid[row, col]) for row, col in coords]
            toa_values = scale_toa_values(toa_values_raw, scale=toa_scale)

            try:
                tot_matrix = render_component_matrix(coords, tot_values, target_size=target_size, dtype=np.int32)
                toa_matrix = render_component_matrix(coords, toa_values, target_size=target_size, dtype=np.int64)  # Updated to np.int64
            except ValueError:
                stats["skipped_tracks"] += 1
                continue

            tot_filename = build_output_name(base_stem, index, "ToT")
            toa_filename = build_output_name(base_stem, index, "ToA")

            save_matrix(tot_matrix, output_tot_dir / rel_dir / tot_filename)
            save_matrix(toa_matrix, output_toa_dir / rel_dir / toa_filename)
            stats["saved_tracks"] += 1

    return stats

In [56]:
# 在确认路径配置正确后取消注释以下代码以启动批处理
stats = process_detector_directory(INPUT_DIR, OUTPUT_TOT_DIR, OUTPUT_TOA_DIR)
stats

{'paired_files': 3238,
 'saved_tracks': 1552,
 'skipped_tracks': 3005,
 'missing_channels': 0}